# 03 — U-Net 학습 (Week 2 PoC)

**목표**: CelebAMask-HQ subset 5K장에서 U-Net 1 epoch 학습 → val mIoU 측정 → 슬라이드용 증명 확보.

**전제**:
- 런타임 → T4 GPU 활성화
- Drive에 cv-final 폴더 업로드 완료 (`project/segmentation/` 최신 버전)

**예상 소요**: 15~20분
- 데이터셋 다운로드: 1.42GB, 약 2분 (캐시되면 즉시)
- 페어 인덱스 빌드: 첫 실행 1~2분 (552K rows 스캔, 이후 캐싱)
- 학습: 1 epoch ~1~3분 (5K subset, T4 GPU)

**Week 2 DoD**: val mIoU > 0 + checkpoint 저장 + 예측 시각화 5장

## 1. 환경 셋업

In [ ]:
!nvidia-smi

In [ ]:
!pip install --quiet segmentation-models-pytorch albumentations datasets pyyaml

In [ ]:
import torch, segmentation_models_pytorch as smp, albumentations as A
from datasets import load_dataset
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available())
print('smp:', smp.__version__)
print('albumentations:', A.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/cv-final'
import sys
sys.path.insert(0, PROJECT_ROOT)

## 2. 데이터셋 — `limsanky/celebamask-hq-256` (Webdataset 포맷)

이 데이터셋은 **552K rows의 flat 구조** (HF webdataset 형식).  
`__key__` 경로로 폴더가 구분됨:

```
images/{N}.png           — 원본 얼굴 (~30K, 256×256 RGB)
mask-anno-256/{N}.png    — 라벨맵 (~30K, 단일채널 0~18) ★
train_color_label/{N}    — 컬러 시각화 (사용 X)
test_color_label/{N}     — 컬러 시각화 (사용 X)
mask-anno/{0~14}/{N}_*   — per-class binary (사용 X)
```

**전략**: `images/{N}` ↔ `mask-anno-256/{N}` 정수 ID로 페어링 → 30K 페어 → 80/10/10 split (시드 42).  
→ `project/segmentation/data.py`가 자동 처리. 첫 실행 시 인덱스 빌드 (1~2분, 이후 캐싱).

In [ ]:
# (선택) 구조 빠른 확인 — 데이터셋 메타
ds_full = load_dataset('limsanky/celebamask-hq-256', split='train')
print(f'total rows: {len(ds_full)}')
print(f'sample key:  {ds_full[0]["__key__"]}')
print(f'sample png:  {type(ds_full[0]["png"]).__name__}, size={ds_full[0]["png"].size}')

In [ ]:
# CelebAMaskHQDataset 검증 (subset 100장)
from project.segmentation import CelebAMaskHQDataset, get_train_transform

ds = CelebAMaskHQDataset(
    split='train',
    transform=get_train_transform(size=256),
    subset_size=100,
)
print()
print('dataset size:', len(ds))
img, mask = ds[0]
print('image:', img.shape, img.dtype)
print('mask:', mask.shape, mask.dtype, 'unique:', mask.unique().tolist())

In [ ]:
# 1배치 시각화 (얼굴 + mask overlay)
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i in range(4):
    img, mask = ds[i]
    img_np = img.permute(1, 2, 0).numpy()
    img_np = img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    img_np = img_np.clip(0, 1)
    axes[i].imshow(img_np)
    axes[i].imshow(mask.numpy(), alpha=0.4, cmap='tab10', vmin=0, vmax=5)
    axes[i].set_title(f'sample {i} · classes: {mask.unique().tolist()}')
    axes[i].axis('off')
plt.tight_layout()
plt.show()

## 3. 학습 실행 (PoC 1 epoch · 5K subset)

In [ ]:
# config.yaml 의 subset_size, epochs 동적 조정
import yaml, os

CONFIG_PATH = f'{PROJECT_ROOT}/project/segmentation/config.yaml'
with open(CONFIG_PATH, 'r') as f:
    cfg = yaml.safe_load(f)

# PoC 설정
cfg['dataset']['subset_size'] = 5000
cfg['train']['epochs'] = 1
cfg['output']['checkpoint_dir'] = f'{PROJECT_ROOT}/project/segmentation/checkpoints'
cfg['dataset']['cache_dir'] = '/content/cv-final-cache'

# checkpoint 폴더 만들기
os.makedirs(cfg['output']['checkpoint_dir'], exist_ok=True)

# 임시 config 저장
TMP_CONFIG = '/tmp/config_poc.yaml'
with open(TMP_CONFIG, 'w') as f:
    yaml.safe_dump(cfg, f)

print('PoC config:')
for section, vals in cfg.items():
    print(f'  {section}: {vals}')

In [ ]:
# 학습 진입
from project.segmentation.train import train

result = train(config_path=TMP_CONFIG)
print('\n=== 학습 완료 ===')
print(f"final mIoU: {result['final_miou']:.4f}")
print(f"final Dice: {result['final_dice']:.4f}")
print(f"checkpoint: {result['checkpoint_path']}")

## 4. 학습 결과 시각화

In [ ]:
# Loss + mIoU curve
epochs = [h['epoch'] for h in result['history']]
losses = [h['loss'] for h in result['history']]
mious = [h['miou'] for h in result['history']]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, losses, marker='o', color='#1E2761')
axes[0].set_title('Training Loss'); axes[0].set_xlabel('epoch'); axes[0].set_ylabel('loss')
axes[0].grid(alpha=0.3)
axes[1].plot(epochs, mious, marker='o', color='#F96167')
axes[1].set_title('Validation mIoU'); axes[1].set_xlabel('epoch'); axes[1].set_ylabel('mIoU')
axes[1].grid(alpha=0.3)
plt.tight_layout()
os.makedirs(f'{PROJECT_ROOT}/data/outputs', exist_ok=True)
plt.savefig(f'{PROJECT_ROOT}/data/outputs/unet_training_curve.png', dpi=150)
plt.show()

In [ ]:
# 예측 시각화 5장 (Input / Ground Truth / Prediction)
from project.segmentation.inference import load_checkpoint
from project.segmentation import get_val_transform

model = load_checkpoint(result['checkpoint_path'])
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# 시각화용 deterministic transform 적용한 검증 데이터셋
ds_viz = CelebAMaskHQDataset(
    split='val',
    transform=get_val_transform(size=256),
    subset_size=5,
)

fig, axes = plt.subplots(5, 3, figsize=(12, 18))
for i in range(5):
    img_tensor, gt_mask = ds_viz[i]
    with torch.no_grad():
        logits = model(img_tensor.unsqueeze(0).to(device))
        pred = logits.argmax(dim=1).squeeze(0).cpu().numpy()

    img_np = img_tensor.permute(1, 2, 0).numpy()
    img_np = img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    img_np = img_np.clip(0, 1)

    axes[i, 0].imshow(img_np); axes[i, 0].set_title('Input'); axes[i, 0].axis('off')
    axes[i, 1].imshow(gt_mask.numpy(), cmap='tab10', vmin=0, vmax=5)
    axes[i, 1].set_title('Ground Truth'); axes[i, 1].axis('off')
    axes[i, 2].imshow(pred, cmap='tab10', vmin=0, vmax=5)
    axes[i, 2].set_title('U-Net Prediction'); axes[i, 2].axis('off')
plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/data/outputs/unet_predictions.png', dpi=150)
plt.show()

## 5. 완료 체크리스트

- [ ] `nvidia-smi`에 T4 표시
- [ ] `pip install` 모든 라이브러리 성공
- [ ] HF 데이터셋 다운로드 완료 (1.42GB)
- [ ] 페어 인덱스 빌드 완료 (`[Dataset] built + cached`)
- [ ] Dataset `__getitem__` 정상 (mask unique 0~4 또는 0~5)
- [ ] 1배치 시각화에서 얼굴 + 라벨 overlay 보임
- [ ] 학습 1 epoch 완료 (loss 감소 확인)
- [ ] val mIoU > 0 (1 epoch이라 0.2~0.4 정도면 OK)
- [ ] checkpoint `unet_v1.pt` 저장됨
- [ ] 예측 시각화 5장 저장됨 → `data/outputs/unet_predictions.png`

**다음 단계 (Week 3)**:
- 본 학습: `subset_size: null` + `epochs: 10` → 24K 전체로 약 50분
- Ablation: augmentation 단계별 비교 (baseline / +flip / +jitter)
- Grad-CAM 시각화 (notebook 04)